### Hito 1 — Baseline & Temporal Validation
### IIT414W F1 Race Strategy Advisor · Team Feligma

## 1. Reproducibility Header & Imports

In [1]:
import sys, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import (
    brier_score_loss,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

RANDOM_SEED = 414
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.size": 12,
    "axes.titlesize": 14,
})

print(f"Python : {sys.version.split()[0]}")
print(f"NumPy  : {np.__version__}")
print(f"Pandas : {pd.__version__}")
print(f"Seed   : {RANDOM_SEED}")

Python : 3.13.5
NumPy  : 2.4.4
Pandas : 2.3.3
Seed   : 414


In [4]:
# Import the dataset
csv_path = "f1_strategy_race_level.csv"
df = pd.read_csv(csv_path)

print(f"Dataset shape: {df.shape}")
print(f"Total rows: {df.shape[0]} races")
print(f"Total columns: {df.shape[1]}")

Dataset shape: (2447, 47)
Total rows: 2447 races
Total columns: 47


In [5]:
# Analyze dataset structure and columns
print("="*80)
print("DATASET COLUMNS (All {})".format(df.shape[1]))
print("="*80)
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col:30s} | dtype: {str(df[col].dtype):10s} | nulls: {df[col].isna().sum():4d}")

print("\n" + "="*80)
print("FIRST 5 ROWS (Preview)")
print("="*80)
print(df.head())

print("\n" + "="*80)
print("DATA TYPES SUMMARY")
print("="*80)
print(df.dtypes)

print("\n" + "="*80)
print("MISSING VALUES")
print("="*80)
missing = df.isnull().sum()
print(missing[missing > 0])

print("\n" + "="*80)
print("BASIC STATISTICS")
print("="*80)
print(df.describe())

DATASET COLUMNS (All 47)
 1. season                         | dtype: int64      | nulls:    0
 2. round                          | dtype: int64      | nulls:    0
 3. race_name                      | dtype: object     | nulls:    0
 4. circuit_id                     | dtype: object     | nulls:    0
 5. circuit                        | dtype: object     | nulls:    0
 6. circuit_type                   | dtype: object     | nulls:    0
 7. driver_id                      | dtype: object     | nulls:    0
 8. driver_name                    | dtype: object     | nulls:    0
 9. Driver                         | dtype: object     | nulls:    0
10. Team                           | dtype: object     | nulls:    0
11. constructor_name               | dtype: object     | nulls:    0
12. grid_position                  | dtype: float64    | nulls:    0
13. qualifying_position            | dtype: float64    | nulls:    0
14. qualifying_time_s              | dtype: float64    | nulls: 2447
15. drive

In [6]:
# Anti-Leakage Analysis: Identify features and avoid post-race observations

print("="*80)
print("FEATURE CATEGORIZATION FOR LEAKAGE PREVENTION")
print("="*80)

# POST-RACE (Leakage): Do NOT use in training
LEAKAGE_FEATURES = [
    'n_stops', 'compound_sequence', 'stint_lengths', 
    'stint1_length', 'stint2_length', 'stint3_length', 'stint4_length', 'stint5_length',
    'avg_pit_stop_duration_s', 'total_pit_time_s', 'first_pit_lap', 'last_pit_lap',
    'finish_position', 'points', 'positions_gained', 'is_top3', 'is_top5', 'is_top10', 'dnf', 'status',
    'wet_laps', 'track_status_summary', 'safety_car_periods', 'safety_car_laps', 'vsc_laps'
]

# PRE-RACE (Safe): Available before race
PRE_RACE_FEATURES = [
    'season', 'round', 'race_name', 'circuit_id', 'circuit', 'circuit_type',
    'driver_id', 'driver_name', 'Driver', 'Team', 'constructor_name',
    'grid_position', 'qualifying_position', 'qualifying_time_s',
    'driver_prior3_avg_finish', 'constructor_prior3_avg_finish',
    'driver_circuit_prior_avg', 'constructor_tier',
    'weather_actual', 'avg_track_temp', 'avg_air_temp'
]

print("\n📌 POST-RACE FEATURES (⚠️ LEAKAGE - DO NOT USE IN TRAINING):")
for i, feat in enumerate(LEAKAGE_FEATURES, 1):
    if feat in df.columns:
        print(f"  ❌ {feat}")

print("\n✅ PRE-RACE FEATURES (SAFE - Use for baseline model):")
for i, feat in enumerate(PRE_RACE_FEATURES, 1):
    if feat in df.columns:
        print(f"  ✓ {feat}")

# Identify baseline features (intersection with what exists)
BASELINE_FEATURES = ['grid_position', 'constructor_tier', 'driver_prior3_avg_finish', 'circuit_type']

print("\n" + "="*80)
print("BASELINE MODEL FEATURES (Final)")
print("="*80)
for feat in BASELINE_FEATURES:
    if feat in df.columns:
        print(f"✓ {feat}: {df[feat].dtype} | nulls: {df[feat].isna().sum()}")
    else:
        print(f"❌ {feat}: NOT FOUND IN DATASET")

# Verify target variable
print("\n" + "="*80)
print("TARGET VARIABLE")
print("="*80)
if 'is_top10' in df.columns:
    print(f"✓ is_top10 (target): {df['is_top10'].dtype}")
    print(f"  Class distribution: \n{df['is_top10'].value_counts().sort_index()}")
    print(f"  Class balance: {df['is_top10'].sum() / len(df) * 100:.2f}% top-10 finishes")

FEATURE CATEGORIZATION FOR LEAKAGE PREVENTION

📌 POST-RACE FEATURES (⚠️ LEAKAGE - DO NOT USE IN TRAINING):
  ❌ n_stops
  ❌ compound_sequence
  ❌ stint_lengths
  ❌ stint1_length
  ❌ stint2_length
  ❌ stint3_length
  ❌ stint4_length
  ❌ stint5_length
  ❌ avg_pit_stop_duration_s
  ❌ total_pit_time_s
  ❌ first_pit_lap
  ❌ last_pit_lap
  ❌ finish_position
  ❌ points
  ❌ positions_gained
  ❌ is_top3
  ❌ is_top5
  ❌ is_top10
  ❌ dnf
  ❌ status
  ❌ wet_laps
  ❌ track_status_summary
  ❌ safety_car_periods
  ❌ safety_car_laps
  ❌ vsc_laps

✅ PRE-RACE FEATURES (SAFE - Use for baseline model):
  ✓ season
  ✓ round
  ✓ race_name
  ✓ circuit_id
  ✓ circuit
  ✓ circuit_type
  ✓ driver_id
  ✓ driver_name
  ✓ Driver
  ✓ Team
  ✓ constructor_name
  ✓ grid_position
  ✓ qualifying_position
  ✓ qualifying_time_s
  ✓ driver_prior3_avg_finish
  ✓ constructor_prior3_avg_finish
  ✓ driver_circuit_prior_avg
  ✓ constructor_tier
  ✓ weather_actual
  ✓ avg_track_temp
  ✓ avg_air_temp

BASELINE MODEL FEATURES (Fin

In [7]:
# Temporal data split (Walk-forward validation)
print("="*80)
print("TEMPORAL DATA SPLIT")
print("="*80)

# Verify season column exists
if 'season' not in df.columns:
    print("❌ ERROR: 'season' column not found in dataset!")
else:
    # Define temporal split per brief specifications
    TRAIN_SEASONS = [2019, 2020, 2021]
    CALIBRATION_SEASONS = [2022]
    TEST_SEASONS = [2023, 2024]
    
    # Split data
    df_train = df[df['season'].isin(TRAIN_SEASONS)].copy()
    df_cal = df[df['season'].isin(CALIBRATION_SEASONS)].copy()
    df_test = df[df['season'].isin(TEST_SEASONS)].copy()
    
    print(f"\n📊 TRAINING SET (Seasons {TRAIN_SEASONS}):")
    print(f"  Rows: {len(df_train)} | Seasons covered: {sorted(df_train['season'].unique())}")
    print(f"  Top-10 class: {df_train['is_top10'].sum() / len(df_train) * 100:.2f}%")
    
    print(f"\n📊 CALIBRATION SET (Seasons {CALIBRATION_SEASONS}):")
    print(f"  Rows: {len(df_cal)} | Seasons covered: {sorted(df_cal['season'].unique())}")
    print(f"  Top-10 class: {df_cal['is_top10'].sum() / len(df_cal) * 100:.2f}%")
    
    print(f"\n📊 TEST SET (Seasons {TEST_SEASONS}):")
    print(f"  Rows: {len(df_test)} | Seasons covered: {sorted(df_test['season'].unique())}")
    print(f"  Top-10 class: {df_test['is_top10'].sum() / len(df_test) * 100:.2f}%")
    
    print(f"\n📌 Total: {len(df_train) + len(df_cal) + len(df_test)} rows (100%)")
    print("="*80)
    print("✅ TEMPORAL SPLIT COMPLETE - No data leakage across folds")

TEMPORAL DATA SPLIT

📊 TRAINING SET (Seasons [2019, 2020, 2021]):
  Rows: 1132 | Seasons covered: [np.int64(2019), np.int64(2020), np.int64(2021)]
  Top-10 class: 51.50%

📊 CALIBRATION SET (Seasons [2022]):
  Rows: 426 | Seasons covered: [np.int64(2022)]
  Top-10 class: 51.64%

📊 TEST SET (Seasons [2023, 2024]):
  Rows: 889 | Seasons covered: [np.int64(2023), np.int64(2024)]
  Top-10 class: 51.74%

📌 Total: 2447 rows (100%)
✅ TEMPORAL SPLIT COMPLETE - No data leakage across folds
